# Partie I : Nettoyage et Préparation des Données

Le fichier `vgsales.csv` contient les ventes de jeux vidéo, en millions d'exemplaires, selon la plateforme et la région. Avant d'analyser ces ventes, on met le fichier au propre : structure, valeurs manquantes, types et doublons.

## 1. Charger et examiner la structure du dataset

In [1]:
import pandas as pd

df = pd.read_csv("vgsales.csv")

print("Nombre de lignes et de colonnes :", df.shape)
df.head(10)

Nombre de lignes et de colonnes : (16598, 11)


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37
5,6,Tetris,GB,1989.0,Puzzle,Nintendo,23.20,2.26,4.22,0.58,30.26
6,7,New Super Mario Bros.,DS,2006.0,Platform,Nintendo,11.38,9.23,6.50,2.90,30.01
7,8,Wii Play,Wii,2006.0,Misc,Nintendo,14.03,9.20,2.93,2.85,29.02
8,9,New Super Mario Bros. Wii,Wii,2009.0,Platform,Nintendo,14.59,7.06,4.70,2.26,28.62
9,10,Duck Hunt,NES,1984.0,Shooter,Nintendo,26.93,0.63,0.28,0.47,28.31


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


In [3]:
df.describe()

,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


Le dataset contient **16 598 jeux** et **11 colonnes**.

- `Rank` : classement du jeu selon les ventes mondiales
- `Name` : titre du jeu
- `Platform` : console
- `Year` : année de sortie
- `Genre` : genre
- `Publisher` : éditeur
- `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`, `Global_Sales` : ventes en millions d'exemplaires

`Rank` est un entier et les ventes sont des décimaux, ce qui est cohérent. `Year` est en float (`2006.0`) : on verra que cela vient des valeurs manquantes. Les quatre autres colonnes sont du texte.

## 2. Identifier et traiter les valeurs manquantes

In [4]:
manquantes = df.isna().sum()
pourcentage = (df.isna().mean() * 100).round(2)

bilan = pd.DataFrame({
    "Valeurs manquantes": manquantes,
    "Pourcentage (%)": pourcentage
})

bilan[bilan["Valeurs manquantes"] > 0]

,Valeurs manquantes,Pourcentage (%)
Year,271,1.63
Publisher,58,0.35


Deux colonnes sont incomplètes :

- `Year` : 271 valeurs, soit 1,63 % des lignes
- `Publisher` : 58 valeurs, soit 0,35 % des lignes

Pour l'éditeur, on remplace le vide par `Inconnu`. Il y a peu de cas, et on conserve les ventes de ces jeux.

Pour l'année, on supprime les lignes. On ne peut pas la retrouver de façon fiable avec les autres colonnes, et elle servira à suivre les ventes dans le temps. La perte reste limitée, moins de 2 % du fichier.

In [5]:
df["Publisher"] = df["Publisher"].fillna("Inconnu")
df = df.dropna(subset=["Year"]).copy()

print("Lignes restantes :", len(df))
df.isna().sum()

Lignes restantes : 16327


Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64

## 3. Vérifier et corriger les types de données

`Year` était en float à cause des valeurs manquantes. Une fois la colonne complète, on la passe en entier.

On retire aussi les espaces en début et en fin de texte, pour éviter qu'un même nom soit compté deux fois à cause d'un espace.

In [6]:
df["Year"] = df["Year"].astype(int)

colonnes_texte = ["Name", "Platform", "Genre", "Publisher"]
for col in colonnes_texte:
    df[col] = df[col].str.strip()

df.dtypes

Rank              int64
Name             object
Platform         object
Year              int64
Genre            object
Publisher        object
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object

## 4. Détecter et supprimer les doublons

In [7]:
print("Doublons exacts :", df.duplicated().sum())

cles = ["Name", "Platform", "Year"]
print("Doublons sur nom + plateforme + année :", df.duplicated(subset=cles).sum())

df[df.duplicated(subset=cles, keep=False)].sort_values(cles)

Doublons exacts : 0
Doublons sur nom + plateforme + année : 1


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
603,604,Madden NFL 13,PS3,2012,Sports,Electronic Arts,2.11,0.23,0.0,0.22,2.56
16127,16130,Madden NFL 13,PS3,2012,Sports,Electronic Arts,0.00,0.01,0.0,0.00,0.01


Aucune ligne n'est recopiée à l'identique. En revanche, *Madden NFL 13* sur PS3 en 2012 est présent deux fois : une ligne à 2,56 millions de ventes, et une autre à 0,01 million.

On retient qu'un jeu correspond à un triplet nom, plateforme et année. On garde la première ligne, celle du classement d'origine, qui porte les ventes les plus élevées.

In [8]:
avant = len(df)
df = df.drop_duplicates(subset=["Name", "Platform", "Year"], keep="first").copy()

print("Lignes supprimées :", avant - len(df))
print("Doublons restants :", df.duplicated(subset=["Name", "Platform", "Year"]).sum())

Lignes supprimées : 1
Doublons restants : 0


## 5. Préparer le dataset pour l'analyse

Le rang d'origine n'est plus continu après les suppressions (le fichier allait jusqu'au rang 16 600). On retrie les jeux par ventes mondiales et on recalcule `Rank`.

On contrôle aussi les années récentes. Le fichier est surtout renseigné jusqu'en 2016 : au-delà, il ne reste que quelques titres. On les retire pour ne pas fausser une analyse par année.

In [9]:
print(df["Year"].value_counts().sort_index().tail(8))

df = df[df["Year"] <= 2016].copy()
df = df.sort_values("Global_Sales", ascending=False).reset_index(drop=True)
df["Rank"] = df.index + 1

df.to_csv("vgsales_clean.csv", index=False)

print("Dimensions finales :", df.shape)
print("Années :", df["Year"].min(), "->", df["Year"].max())
df.head()

Year
2011    1139
2012     656
2013     546
2014     582
2015     614
2016     344
2017       3
2020       1
Name: count, dtype: int64
Dimensions finales : (16322, 11)
Années : 1980 -> 2016


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16322 entries, 0 to 16321
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16322 non-null  int64  
 1   Name          16322 non-null  object 
 2   Platform      16322 non-null  object 
 3   Year          16322 non-null  int64  
 4   Genre         16322 non-null  object 
 5   Publisher     16322 non-null  object 
 6   NA_Sales      16322 non-null  float64
 7   EU_Sales      16322 non-null  float64
 8   JP_Sales      16322 non-null  float64
 9   Other_Sales   16322 non-null  float64
 10  Global_Sales  16322 non-null  float64
dtypes: float64(5), int64(2), object(4)
memory usage: 1.4+ MB


Le fichier nettoyé est enregistré dans `vgsales_clean.csv`.

Il contient **16 322 jeux**, sortis entre **1980 et 2016** :

- les éditeurs manquants sont codés `Inconnu`
- les jeux sans année ont été retirés
- `Year` est un entier
- chaque couple jeu / plateforme / année n'apparaît qu'une fois
- `Rank` suit de nouveau l'ordre des ventes mondiales

Le dataset est prêt pour l'analyse.